### Code to process data in folders

#### Generate latex rows

In [ ]:
import pandas as pd
import re
import os
from collections import defaultdict

def extract_mean_std(value):
    """Handles both '0.027 ± 0.003' and '0.027' formats."""
    if isinstance(value, str) and '±' in value:
        match = re.match(r"\s*([\d\.Ee+-]+)\s*[±±]\s*([\d\.Ee+-]+)", value)
        if match:
            mean, std = match.groups()
            return float(mean), float(std)
    try:
        # If it's just a number (or numeric string)
        mean = float(value)
        return mean, None
    except:
        return None, None

def generate_latex_alternating_row(csv_path):
    df = pd.read_csv(csv_path)
    if df.shape[1] < 3:
        raise ValueError("CSV must have at least 3 columns: model, MAE, RMSE")

    latex_rows = []
    for _, row in df.iterrows():
        row_cells = [row.iloc[0]]  # model/target name
        for col in df.columns[1:3]:  # MAE and RMSE
            mean, std = extract_mean_std(row[col])
            if mean is None:
                formatted = "-"
            elif std is None:
                formatted = f"{mean:.3f}"
            else:
                formatted = f"{mean:.3f}{{\\scriptsize±{std:.3f}}}"
            row_cells.append(formatted)
        latex_rows.append(" & ".join(row_cells))
    return latex_rows

sampling_rate = 25
# Model folder names (replace these with your actual 5 model folders)
model_names = [
    "TimeGPT",
    "TimesFM",
    "MOIRAI",
    "Chronos",
    "TimerXL",
]

# Rename mapping
rename_map = {
    "london_load": "London",
    "zonnedael_customer_8": "Cust 8",
    "zonnedael_customer_9": "Cust 9",
    "zonnedael_customer_43": "Cust 43"
}

# rename_map = {
#     "pv_house_1": "PV 1",
#     "pv_house_2": "PV 2",
#     "pv_house_3": "PV 3",
#     "pv_house_4": "PV 4",
#     "bess_house_1": "BESS 1",
#     "bess_house_2": "BESS 2",
#     "bess_house_3": "BESS 3",
#     "bess_house_4": "BESS 4",
#     "load": "Load"
# }

# Base path pattern
base_path = "results/results_belgium" 

# Collect all rows from all models
all_model_outputs = defaultdict(list)
target_order = []

for model_name in model_names:
    # csv_file = os.path.join(base_path, model_name, f"Sampling_{sampling_rate}", f"Sampling_{sampling_rate}_model_metrics_average.csv")
    csv_file = os.path.join(base_path, model_name, f"Sampling_{sampling_rate}", "Run_1", "model_metrics_summary.csv")
    try:
        latex_rows = generate_latex_alternating_row(csv_file)
        for row in latex_rows:
            parts = [p.strip() for p in row.split("&")]
            if len(parts) != 3:
                continue
            target, mae, rmse = parts
            all_model_outputs[target].append(mae)
            all_model_outputs[target].append(rmse)
            if target not in target_order:
                target_order.append(target)
    except Exception as e:
        print(f"Error with {model_name}: {e}")

# Output LaTeX table rows
for target in target_order:
    renamed = rename_map.get(target, target)
    values = all_model_outputs[target]
    print(f"{renamed} & " + " & ".join(values) + r" \\")


#### Move to Sampling_100 folder

In [ ]:
import os
import shutil

# Define the root results directory
root_results_dir = "results"

# Loop through each model folder inside the results directory
for model_name in os.listdir(root_results_dir):
    model_path = os.path.join(root_results_dir, model_name)

    # Skip if not a directory
    if not os.path.isdir(model_path):
        continue

    # Define the Sampling_100 path
    sampling_100_path = os.path.join(model_path, "Sampling_100")

    # Create Sampling_100 folder if it does not exist
    if not os.path.exists(sampling_100_path):
        os.makedirs(sampling_100_path)
        print(f"Created folder: {sampling_100_path}")

    # Move all subfolders (not files) into Sampling_100
    for item in os.listdir(model_path):
        item_path = os.path.join(model_path, item)

        # Skip if it's the Sampling_100 folder or a file
        if item == "Sampling_100" or not os.path.isdir(item_path):
            continue

        # Destination path inside Sampling_100
        dest_path = os.path.join(sampling_100_path, item)

        # Move the directory
        shutil.move(item_path, dest_path)
        print(f"Moved {item_path} → {dest_path}")

#### Mean +/- std

In [ ]:
import os
import pandas as pd

# Base directory containing results
base_dir = "results/results_london_zonnedael"
n_epochs = 100
sampling_rate = 100 # 25, 33, 50 or 100

# Automatically detect all model directories
model_names = [
    name for name in os.listdir(base_dir)
    if os.path.isdir(os.path.join(base_dir, name)) and 
       os.path.exists(os.path.join(base_dir, name, f"Sampling_{sampling_rate}"))
]

for model in model_names:
    sampling_dir = os.path.join(base_dir, model, f"Sampling_{sampling_rate}")
    epoch_dirs = [os.path.join(sampling_dir, f"Epochs_{n_epochs}_{i}") for i in range(1, 11)]

    all_metrics = []
    reference_order = None  # To store the first file's model order

    for epoch_dir in epoch_dirs:
        csv_path = os.path.join(epoch_dir, "model_metrics_summary.csv")
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            all_metrics.append(df)
            if reference_order is None:
                reference_order = df["model"].tolist()
        else:
            print(f"Missing: {csv_path}")

    if not all_metrics:
        print(f"No valid CSVs found for {model}")
        continue

    # Concatenate all metric DataFrames
    combined_df = pd.concat(all_metrics, ignore_index=True)

    # Compute mean ± std per model while preserving order from first CSV
    summary_rows = []
    for model_name in reference_order:
        group_df = combined_df[combined_df["model"] == model_name]
        row = {"model": model_name}
        for col in ["MAE", "RMSE", "MAPE", "R2"]:
            mean = group_df[col].mean()
            std = group_df[col].std()
            row[col] = f"{mean:.6f} ± {std:.6f}"
        summary_rows.append(row)

    # Create summary DataFrame
    summary_df = pd.DataFrame(summary_rows)

    # Save to CSV
    output_path = os.path.join(sampling_dir, f"Sampling_{sampling_rate}_model_metrics_average.csv")
    summary_df.to_csv(output_path, index=False)
    print(f"Saved summary for {model} to {output_path}")


#### Mean +/- std for LightGBM

In [ ]:
import os
import pandas as pd
import numpy as np

# Base directory containing results
base_dir = "results/results_belgium"

# List of model folders to process
model_names = ["Mamba"]
sampling_rate = 100

for model in model_names:
    sampling_dir = os.path.join(base_dir, model, f"Sampling_{sampling_rate}")
    epoch_dirs = [os.path.join(sampling_dir, f"Run_{i}") for i in range(1, 11)]

    all_metrics = []
    reference_order = None  # To store the first file's model order

    for epoch_dir in epoch_dirs:
        csv_path = os.path.join(epoch_dir, "model_metrics_summary.csv")
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            all_metrics.append(df)
            if reference_order is None:
                reference_order = df["model"].tolist()
        else:
            print(f"Missing: {csv_path}")

    if not all_metrics:
        print(f"No valid CSVs found for {model}")
        continue

    # Concatenate all metric DataFrames
    combined_df = pd.concat(all_metrics, ignore_index=True)

    # Compute mean ± std per model while preserving order from first CSV
    summary_rows = []
    for model_name in reference_order:
        group_df = combined_df[combined_df["model"] == model_name]
        row = {"model": model_name}
        for col in ["MAE", "RMSE", "MAPE", "R2"]:
            mean = group_df[col].mean()
            std = group_df[col].std()
            row[col] = f"{mean:.6f} ± {std:.6f}"
        summary_rows.append(row)

    # Create summary DataFrame
    summary_df = pd.DataFrame(summary_rows)

    # Save to CSV
    output_path = os.path.join(sampling_dir, f"Sampling_{sampling_rate}_model_metrics_average.csv")
    summary_df.to_csv(output_path, index=False)
    print(f"Saved summary for {model} to {output_path}")


#### Rename files

In [ ]:
import os

# Path to the directory containing your files
folder_path = "results/results_london_zonnedael/AutoARIMA/Sampling_100/Run_1"

# Loop through all files in the directory
for filename in os.listdir(folder_path):
    if "_AutoARIMA" in filename:
        new_filename = filename.replace("_AutoARIMA", "")
        old_file = os.path.join(folder_path, filename)
        new_file = os.path.join(folder_path, new_filename)
        os.rename(old_file, new_file)
        print(f"Renamed: {filename} -> {new_filename}")


#### Move folders to recycle bin 

In [ ]:
import os
from send2trash import send2trash

# Root results directory
base_path = "results"

# Loop through each model folder
for model_name in os.listdir(base_path):
    model_path = os.path.join(base_path, model_name)
    delete_path = os.path.join(model_path, "Sampling_75")

    if os.path.isdir(delete_path):
        print(f"Sending to Recycle Bin: {delete_path}")
        send2trash(delete_path)


### Plot forecast results

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Consistent font size
font_size = 12
tick_font_size = font_size - 2
legend_font_size = font_size - 4

plt.rcParams.update({
    'font.size': font_size,
    'axes.labelsize': font_size,
    'axes.titlesize': font_size,
    'xtick.labelsize': tick_font_size,
    'ytick.labelsize': tick_font_size,
    'legend.fontsize': legend_font_size,
    'lines.linewidth': 2,
})

all_models = [
    "KNNRegression", "LightGBM", "AutoARIMA", "NaiveDrift", "NaiveMovingAverage",
    "TCN", "BiTCN", "TimesNet", "WaveNet", "MQCNN",
    "NHITS", "NBEATS", "TiDE", "NLinear", "DeepNPTS",
    "DeepFactor", "DeepAR", "Mamba", "TemporalFusionTransformer", "MQRNN",
    "Informer", "PatchTST", "iTransformer", "VanillaTransformer", "TimeXer",
    "TimeGPT", "TimesFM", "MOIRAI", "Chronos", "TimerXL"
]
letters = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)']

csv_filenames = ["pv_house_1_forecast_vs_actual.csv", # 1-4
                "bess_house_1_forecast_vs_actual.csv", # 1-4
                "zonnedael_customer_8_forecast_vs_actual.csv", # 8, 9, 43
                "load_forecast_vs_actual.csv",
                "london_load_forecast_vs_actual.csv"
                ]
csv_filename = "london_load_forecast_vs_actual.csv"

if csv_filename.startswith("pv") or csv_filename.startswith("bess") or csv_filename.startswith("load"):
    base_dir = "results/results_belgium"
else:
    base_dir = "results/results_london_zonnedael"

legend_name_map = {
    "KNNRegression": "KNN Regression",
    "NaiveDrift": "Naive Drift",
    "NaiveMovingAverage": "Naive MA",
    "MQCNN": "MQ-CNN",
    "TemporalFusionTransformer": "TFT",
    "MQRNN": "MQ-RNN",
    "VanillaTransformer": "Vanilla Transformer",
    "TimerXL": "Timer-XL"
}

model_groups = [all_models[i:i + 5] for i in range(0, 30, 5)]
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(14, 6))
axes = axes.flatten()

xtick_locs = [0, 24, 48, 72, 96, 120, 144, 168, 192]
xtick_labels = ["00:00", "06:00", "12:00", "18:00", "00:00", "06:00", "12:00", "18:00", "00:00"]

for idx, (models, letter) in enumerate(zip(model_groups, letters)):
    ax = axes[idx]

    deepar_index = all_models.index("DeepAR")
    if deepar_index < 5 or deepar_index >= 25:
        deepar_subfolder = os.path.join("Sampling_100", "Run_1")
    else:
        deepar_subfolder = os.path.join("Sampling_100", "Epochs_100_1")

    deepar_path = os.path.join(base_dir, "DeepAR", deepar_subfolder, csv_filename)
    if os.path.exists(deepar_path):
        df_deepar = pd.read_csv(deepar_path)
        ax.plot(df_deepar["Actual"], label="Actual", color='black', linestyle='-', linewidth=1.5)
        ax.set_xlim(0, len(df_deepar["Actual"]))
    else:
        print(f"Missing: {deepar_path}")

    for model in models:
        model_index = all_models.index(model)
        if model_index < 5 or model_index >= 25:
            subfolder = os.path.join("Sampling_100", "Run_1")
        else:
            subfolder = os.path.join("Sampling_100", "Epochs_100_1")

        model_path = os.path.join(base_dir, model, subfolder, csv_filename)
        if not os.path.exists(model_path):
            print(f"Missing: {model_path}")
            continue

        df = pd.read_csv(model_path)
        display_name = legend_name_map.get(model, model)
        ax.plot(df["Forecast"], label=display_name, linestyle='-', linewidth=1.5)

    ax.set_title("")
    ax.text(-0.11, 1.1, letter, transform=ax.transAxes,
            fontsize=font_size, fontweight='normal', ha='left', va='top')

    ax.grid(True)
    ax.tick_params(axis='both', which='major', labelsize=tick_font_size)

    if csv_filename.startswith("pv"):
        ax.set_ylabel("PV Power", fontsize=font_size)
    elif csv_filename.startswith("bess"):
        ax.set_ylabel("BESS Power", fontsize=font_size)
    elif csv_filename.startswith("zonnedael"):
        ax.set_ylabel("Load", fontsize=font_size)
    else:
        ax.set_ylabel("Load", fontsize=font_size)

    if idx < 4:
        ax.set_xticklabels([])
        ax.tick_params(labelbottom=False)
    else:
        ax.tick_params(labelbottom=True)
        ax.set_xlabel("Time", fontsize=font_size)
        ax.set_xticks(xtick_locs)
        ax.set_xticklabels(xtick_labels, rotation=0, fontsize=tick_font_size)

    ax.legend(fontsize=legend_font_size, loc='upper left')

plt.tight_layout()
output_folder = "results/figures"
os.makedirs(output_folder, exist_ok=True)
output_file = os.path.join(output_folder, "results_pv_house_2.pdf")

base_name = csv_filename.replace("_forecast_vs_actual.csv", "")
output_file = os.path.join(output_folder, f"results_{base_name}.pdf")
print(output_file)

plt.savefig(output_file, format='pdf', bbox_inches='tight', dpi=300)
plt.show()
plt.close()

print(f"Saved to {output_file}")


In [ ]:
import os
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

all_models = [
    "KNNRegression", "LightGBM", "AutoARIMA", "NaiveDrift", "NaiveMovingAverage",
    "TCN", "BiTCN", "TimesNet", "WaveNet", "MQCNN",
    "NHITS", "NBEATS", "TiDE", "NLinear", "DeepNPTS",
    "DeepFactor", "DeepAR", "Mamba", "TemporalFusionTransformer", "MQRNN",
    "Informer", "PatchTST", "iTransformer", "VanillaTransformer", "TimeXer",
    "TimeGPT", "TimesFM", "MOIRAI", "Chronos", "TimerXL"
]
letters = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)']

csv_filename = "zonnedael_customer_43_forecast_vs_actual.csv"

if csv_filename.startswith("pv") or csv_filename.startswith("bess"):
    base_dir = "results/results_belgium"
else:
    base_dir = "results/results_london_zonnedael"

legend_name_map = {
    "KNNRegression": "KNN Regression",
    "NaiveDrift": "Naive Drift",
    "NaiveMovingAverage": "Naive MA",
    "MQCNN": "MQ-CNN",
    "TemporalFusionTransformer": "TFT",
    "MQRNN": "MQ-RNN",
    "VanillaTransformer": "Vanilla Transformer",
    "TimerXL": "Timer-XL"
}

model_groups = [all_models[i:i + 5] for i in range(0, 30, 5)]

# Create subplot grid (3x2)
fig = make_subplots(rows=3, cols=2, subplot_titles=letters)

xtick_labels = ["00:00", "06:00", "12:00", "18:00", "00:00", "06:00", "12:00", "18:00", "00:00"]

for idx, (models, letter) in enumerate(zip(model_groups, letters)):
    row = idx // 2 + 1
    col = idx % 2 + 1

    # DeepAR baseline
    deepar_index = all_models.index("DeepAR")
    if deepar_index < 5 or deepar_index >= 25:
        deepar_subfolder = os.path.join("Sampling_100", "Run_1")
    else:
        deepar_subfolder = os.path.join("Sampling_100", "Epochs_100_1")

    deepar_path = os.path.join(base_dir, "DeepAR", deepar_subfolder, csv_filename)
    if os.path.exists(deepar_path):
        df_deepar = pd.read_csv(deepar_path)
        fig.add_trace(
            go.Scatter(
                y=df_deepar["Actual"],
                mode="lines",
                name="Actual",
                line=dict(color="black", width=2),
                legendgroup="Actual"
            ),
            row=row, col=col
        )

    # Forecasts for each model
    for model in models:
        model_index = all_models.index(model)
        if model_index < 5 or model_index >= 25:
            subfolder = os.path.join("Sampling_100", "Run_1")
        else:
            subfolder = os.path.join("Sampling_100", "Epochs_100_1")

        model_path = os.path.join(base_dir, model, subfolder, csv_filename)
        if not os.path.exists(model_path):
            continue

        df = pd.read_csv(model_path)
        display_name = legend_name_map.get(model, model)

        fig.add_trace(
            go.Scatter(
                y=df["Forecast"],
                mode="lines",
                name=display_name,
                legendgroup=display_name,
                showlegend=True
            ),
            row=row, col=col
        )

# Axis labels
yaxis_title = "Load" if csv_filename.startswith("zonnedael") else \
              "PV Power" if csv_filename.startswith("pv") else "BESS Power"

fig.update_yaxes(title_text=yaxis_title)

# Formatting
fig.update_layout(
    height=900,
    width=1200,
    title_text=f"Forecast Results: {csv_filename}",
    hovermode="x unified"
)

# Save as interactive HTML
output_folder = "results/Figures"
os.makedirs(output_folder, exist_ok=True)
base_name = csv_filename.replace("_forecast_vs_actual.csv", "")
output_file = os.path.join(output_folder, f"results_{base_name}.html")

fig.write_html(output_file, include_plotlyjs="cdn")
print(f"Saved interactive plot to {output_file}")
fig.show()
